In [ ]:
import logging
import uuid
from datetime import timedelta
from pathlib import Path

import pendulum
import polars as pl
import requests
import tenacity
import yaml
from sqlalchemy import create_engine, text

from dotenv import load_dotenv
import os
import re

True

### Путь к конфиг файлу

In [2]:
CONFIG_PATH = Path("config.yaml")
project_root = Path("weather_etl")

### Конфиг, логгер, движок БД

In [ ]:
load_dotenv()

def load_settings(cfg_path) -> dict:
    with open(cfg_path, "r", encoding="utf-8") as fh:
        raw = fh.read()
    # заменяет ${DB_URL} → реальное значение из .env
    raw = re.sub(r"\$\{(\w+)\}", lambda m: os.environ.get(m.group(1), ""), raw)
    return yaml.safe_load(raw)

CFG = load_settings(CONFIG_PATH)
TABLE_NAME = f'{CFG["database"]["schema"]}.{CFG["database"]["table"]}'


def build_logger(settings: dict, run_id: str) -> logging.LoggerAdapter:
    log_path = project_root / settings["logging"]["file_name"]
    log_path.parent.mkdir(parents=True, exist_ok=True)

    base_logger = logging.getLogger("weather_etl")
    base_logger.setLevel(getattr(logging, settings["logging"]["level"].upper(), logging.INFO))

    if not base_logger.handlers:
        fmt = logging.Formatter(
            "%(asctime)s | %(levelname)s | etl_id=%(etl_id)s | %(message)s"
        )
        fh = logging.FileHandler(log_path, mode="a", encoding="utf-8")
        fh.setFormatter(fmt)
        sh = logging.StreamHandler()
        sh.setFormatter(fmt)
        base_logger.addHandler(fh)
        base_logger.addHandler(sh)

    return logging.LoggerAdapter(base_logger, extra={"etl_id": run_id})


def db_engine(settings: dict):
    db = settings["database"]
    url = db.get("url")
    if url and "${" not in url:
        return create_engine(url)
    # fallback — через отдельные параметры из .env
    return create_engine(
        "postgresql+psycopg2://",
        connect_args={
            "host":     os.environ.get("DB_HOST", "localhost"),
            "port":     int(os.environ.get("DB_PORT", 5432)),
            "dbname":   os.environ.get("DB_NAME", "weather"),
            "user":     os.environ.get("DB_USER", "postgres"),
            "password": os.environ.get("DB_PASSWORD", ""),
        }
    )

### Создание таблицы в случае её отсутствия

In [4]:
def prepare_database(engine, settings: dict) -> None:
    schema = settings["database"]["schema"]
    table  = settings["database"]["table"]

    ddl = f"""
        CREATE TABLE IF NOT EXISTS {schema}.{table} (
            id              BIGSERIAL PRIMARY KEY,
            city            TEXT          NOT NULL,
            datetime        TIMESTAMPTZ   NOT NULL,
            temperature_2m  FLOAT,
            precipitation   FLOAT,
            windspeed_10m   FLOAT,
            cloudcover      INT,
            loaded_at       TIMESTAMPTZ,
            etl_id          TEXT,
            UNIQUE (city, datetime)
        );
    """

    with engine.begin() as conn:
        conn.execute(text(ddl))
    print("Таблица готова:", f"{schema}.{table}")

### Инкремент

In [5]:
def read_hwm(engine, city: str, settings: dict) -> str:
    """Возвращает дату следующего дня после последней загруженной записи."""
    fallback = settings["fetch"]["start_date_fallback"]

    query = text(
        f"SELECT MAX(datetime::date)::text AS hwm "
        f"FROM {TABLE_NAME} WHERE city = :city"
    )
    with engine.connect() as conn:
        row = conn.execute(query, {"city": city}).fetchone()

    if row is None or row[0] is None:
        return fallback

    last_date = pendulum.parse(row[0])
    return last_date.add(days=1).to_date_string()

### Получение данных

In [6]:
def fetch_weather(city_cfg: dict, start_date: str, end_date: str,
                  settings: dict) -> dict:
    """Запрос к Open-Meteo Historical API с автоматическими ретраями."""

    @tenacity.retry(
        stop=tenacity.stop_after_attempt(settings["retry"]["attempts"]),
        wait=tenacity.wait_fixed(settings["retry"]["wait_seconds"]),
        retry=tenacity.retry_if_exception_type(
            (requests.RequestException, ValueError)
        ),
        reraise=True,
    )
    def _do_request() -> dict:
        url = "https://archive-api.open-meteo.com/v1/archive"
        params = {
            "latitude":  city_cfg["latitude"],
            "longitude": city_cfg["longitude"],
            "start_date": start_date,
            "end_date":   end_date,
            "hourly":     ",".join(settings["fetch"]["hourly_variables"]),
            "timezone":   settings["fetch"]["timezone"],
        }
        headers = {"User-Agent": settings["http"]["user_agent"]}

        resp = requests.get(
            url, params=params, headers=headers,
            timeout=settings["http"]["timeout"]
        )
        resp.raise_for_status()
        payload = resp.json()

        if "hourly" not in payload:
            raise ValueError(f"Нет ключа 'hourly' в ответе: {payload}")

        return payload

    return _do_request()

### Преобразование строк

In [7]:
def parse_rows(payload: dict, city_name: str,
               etl_run_id: str, settings: dict) -> list[dict]:
    """Преобразует массивы Open-Meteo в список словарей (одна строка = 1 час)."""
    hourly    = payload["hourly"]
    times     = hourly["time"]
    variables = settings["fetch"]["hourly_variables"]
    loaded_at = pendulum.now("UTC").to_iso8601_string()

    rows = []
    for i, t in enumerate(times):
        row = {"city": city_name, "datetime": t,
               "loaded_at": loaded_at, "etl_id": etl_run_id}
        for var in variables:
            row[var] = hourly.get(var, [None] * len(times))[i]
        rows.append(row)

    return rows

### Сохранение в БД и Parquet

In [8]:
def persist_dataframe(dataset: pl.DataFrame, engine,
                      settings: dict, logger) -> None:
    if dataset.is_empty():
        logger.info("Новых записей нет, сохранение не требуется.")
        return

    # ---------- БД ----------
    rows_dicts = dataset.to_dicts()
    insert_sql = text(f"""
        INSERT INTO {TABLE_NAME}
            (city, datetime, temperature_2m, precipitation,
             windspeed_10m, cloudcover, loaded_at, etl_id)
        VALUES
            (:city, :datetime, :temperature_2m, :precipitation,
             :windspeed_10m, :cloudcover, :loaded_at, :etl_id)
        ON CONFLICT (city, datetime) DO NOTHING
    """)
    with engine.begin() as conn:
        conn.execute(insert_sql, rows_dicts)

    # ---------- Parquet (колоночный, сжатый) ----------
    export_dir = project_root / settings["filesystem"]["output_dir"]
    export_dir.mkdir(parents=True, exist_ok=True)

    ts   = pendulum.now("UTC").format("YYYYMMDD_HHmmss")
    path = export_dir / f"weather_increment_{ts}.parquet"

    dataset.write_parquet(str(path),
                          compression=settings["filesystem"]["compression"])

    logger.info("Сохранено в БД строк: %s", dataset.height)
    logger.info("Сохранено в Parquet:  %s", path)

### ETL

In [9]:
def run_incremental_etl() -> pl.DataFrame:
    etl_id  = str(uuid.uuid4())
    logger  = build_logger(CFG, etl_id)
    engine  = db_engine(CFG)

    logger.info("ETL-запуск начат.")
    prepare_database(engine, CFG)

    yesterday = pendulum.yesterday("UTC").to_date_string()
    all_rows  = []

    for city_cfg in CFG["cities"]:
        city = city_cfg["name"]
        try:
            start_date = read_hwm(engine, city, CFG)

            if start_date > yesterday:
                logger.info("%s: данные актуальны (%s), пропускаем.", city, start_date)
                continue

            logger.info("%s: загрузка %s → %s", city, start_date, yesterday)

            payload = fetch_weather(city_cfg, start_date, yesterday, CFG)
            rows    = parse_rows(payload, city, etl_id, CFG)

            logger.info("%s: получено строк %s", city, len(rows))
            all_rows.extend(rows)

        except Exception as exc:
            logger.exception("Ошибка при обработке %s: %s", city, exc)
            raise

    if not all_rows:
        logger.info("Источник не вернул новых данных.")
        return pl.DataFrame()

    # ── Сборка датафрейма ──
    df = pl.DataFrame(all_rows)

    df = df.with_columns([
        pl.col("datetime")
        .str.to_datetime(format="%Y-%m-%dT%H:%M", strict=False)
        .dt.replace_time_zone("Europe/Moscow")
        .dt.convert_time_zone("UTC"),

        # берём первые 19 символов (без +00:00) и явно ставим UTC
        pl.col("loaded_at")
        .str.slice(0, 19)
        .str.to_datetime(format="%Y-%m-%dT%H:%M:%S")
        .dt.replace_time_zone("UTC"),
    ])

    # ── Фильтрация уже существующих записей ──
    existing = pl.read_database(
        query=f"SELECT city, datetime FROM {TABLE_NAME}",
        connection=engine,
    )
    if not existing.is_empty():
        df = df.join(existing, on=["city", "datetime"], how="anti")

    persist_dataframe(df, engine, CFG, logger)
    logger.info("ETL-запуск завершён. Новых строк: %s", df.height)

    return df

In [10]:
df_result = run_incremental_etl()
df_result

2026-04-01 21:44:17,843 | INFO | etl_id=5fe21ccd-1aae-4aa4-9ab0-0794f6d13a6a | ETL-запуск начат.
2026-04-01 21:44:18,015 | INFO | etl_id=5fe21ccd-1aae-4aa4-9ab0-0794f6d13a6a | Moscow: загрузка 2026-01-01 → 2026-03-31


Таблица готова: weather.hourly_data


2026-04-01 21:44:18,332 | INFO | etl_id=5fe21ccd-1aae-4aa4-9ab0-0794f6d13a6a | Moscow: получено строк 2160
2026-04-01 21:44:18,334 | INFO | etl_id=5fe21ccd-1aae-4aa4-9ab0-0794f6d13a6a | Saint Petersburg: загрузка 2026-01-01 → 2026-03-31
2026-04-01 21:44:20,422 | INFO | etl_id=5fe21ccd-1aae-4aa4-9ab0-0794f6d13a6a | Saint Petersburg: получено строк 2160
2026-04-01 21:44:20,424 | INFO | etl_id=5fe21ccd-1aae-4aa4-9ab0-0794f6d13a6a | Novosibirsk: загрузка 2026-01-01 → 2026-03-31
2026-04-01 21:44:20,728 | INFO | etl_id=5fe21ccd-1aae-4aa4-9ab0-0794f6d13a6a | Novosibirsk: получено строк 2160
2026-04-01 21:44:21,529 | INFO | etl_id=5fe21ccd-1aae-4aa4-9ab0-0794f6d13a6a | Сохранено в БД строк: 6480
2026-04-01 21:44:21,530 | INFO | etl_id=5fe21ccd-1aae-4aa4-9ab0-0794f6d13a6a | Сохранено в Parquet:  weather_etl\data\weather_increment_20260401_184421.parquet
2026-04-01 21:44:21,531 | INFO | etl_id=5fe21ccd-1aae-4aa4-9ab0-0794f6d13a6a | ETL-запуск завершён. Новых строк: 6480


city,datetime,loaded_at,etl_id,temperature_2m,precipitation,windspeed_10m,cloudcover
str,"datetime[μs, UTC]","datetime[μs, UTC]",str,f64,f64,f64,i64
"""Moscow""",2025-12-31 21:00:00 UTC,2026-04-01 18:44:18 UTC,"""5fe21ccd-1aae-4aa4-9ab0-0794f6…",-7.8,0.0,10.3,100
"""Moscow""",2025-12-31 22:00:00 UTC,2026-04-01 18:44:18 UTC,"""5fe21ccd-1aae-4aa4-9ab0-0794f6…",-7.9,0.0,11.3,100
"""Moscow""",2025-12-31 23:00:00 UTC,2026-04-01 18:44:18 UTC,"""5fe21ccd-1aae-4aa4-9ab0-0794f6…",-8.8,0.0,11.4,100
"""Moscow""",2026-01-01 00:00:00 UTC,2026-04-01 18:44:18 UTC,"""5fe21ccd-1aae-4aa4-9ab0-0794f6…",-10.0,0.0,9.7,66
"""Moscow""",2026-01-01 01:00:00 UTC,2026-04-01 18:44:18 UTC,"""5fe21ccd-1aae-4aa4-9ab0-0794f6…",-10.1,0.0,9.8,100
…,…,…,…,…,…,…,…
"""Novosibirsk""",2026-03-31 16:00:00 UTC,2026-04-01 18:44:20 UTC,"""5fe21ccd-1aae-4aa4-9ab0-0794f6…",-2.8,0.0,1.7,100
"""Novosibirsk""",2026-03-31 17:00:00 UTC,2026-04-01 18:44:20 UTC,"""5fe21ccd-1aae-4aa4-9ab0-0794f6…",-3.4,0.0,1.3,100
"""Novosibirsk""",2026-03-31 18:00:00 UTC,2026-04-01 18:44:20 UTC,"""5fe21ccd-1aae-4aa4-9ab0-0794f6…",-3.2,0.0,2.9,100


In [11]:
df_result = run_incremental_etl()
df_result

2026-04-01 21:55:02,579 | INFO | etl_id=8fe9cd3b-0482-41e2-bec8-71d22fd54609 | ETL-запуск начат.
2026-04-01 21:55:02,703 | INFO | etl_id=8fe9cd3b-0482-41e2-bec8-71d22fd54609 | Moscow: данные актуальны (2026-04-01), пропускаем.
2026-04-01 21:55:02,706 | INFO | etl_id=8fe9cd3b-0482-41e2-bec8-71d22fd54609 | Saint Petersburg: данные актуальны (2026-04-01), пропускаем.
2026-04-01 21:55:02,709 | INFO | etl_id=8fe9cd3b-0482-41e2-bec8-71d22fd54609 | Novosibirsk: данные актуальны (2026-04-01), пропускаем.
2026-04-01 21:55:02,710 | INFO | etl_id=8fe9cd3b-0482-41e2-bec8-71d22fd54609 | Источник не вернул новых данных.


Таблица готова: weather.hourly_data


shape: (0, 0)
┌┐
╞╡
└┘